# XGBoost Model Selection

Fit and evaluate a leakage-safe XGBoost classifier from `diabetes.csv`. The cleaning step from `cleaning_and_scaling.ipynb` is kept inside the pipeline, and hyperparameters are selected with stratified cross-validation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
from xgboost import XGBClassifier

## Cleaning logic checked

This notebook uses the same cleaning idea as `cleaning_and_scaling.ipynb`: zeros are treated as invalid for `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI`, then replaced with each column median.

The difference is that the median replacement happens inside the scikit-learn pipeline. During cross-validation, each fold learns those medians only from its own training data, avoiding leakage into validation or test rows.

In [2]:
raw_data = pd.read_csv("diabetes.csv")
print(raw_data.shape)
raw_data.head()

(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
zero_as_missing_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

raw_data.eq(0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [4]:
class ZeroToMedianImputer(BaseEstimator, TransformerMixin):
    """Replace invalid zeros in selected columns with training medians."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = X.columns.to_numpy()
        self.medians_ = {}

        for column in self.columns:
            non_zero_values = X.loc[X[column] != 0, column]
            self.medians_[column] = non_zero_values.median()

        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()

        for column, median in self.medians_.items():
            X[column] = X[column].replace(0, median)

        return X

In [5]:
X = raw_data.drop(columns="Outcome")
y = raw_data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train class balance:")
print(y_train.value_counts(normalize=True).rename("proportion"))
print("\nTest class balance:")
print(y_test.value_counts(normalize=True).rename("proportion"))

Train class balance:
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test class balance:
0    0.649351
1    0.350649
Name: proportion, dtype: float64


## Cross-validated hyperparameter search

`GridSearchCV` evaluates XGBoost settings with 5-fold stratified CV. Scaling is not used here because boosted trees do not require standardized features.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

pipeline = Pipeline(
    steps=[
        ("zero_imputer", ZeroToMedianImputer(columns=zero_as_missing_columns)),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [2, 3, 4],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__reg_lambda": [1, 5],
    "model__scale_pos_weight": [1, scale_pos_weight],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={"roc_auc": "roc_auc", "f1": "f1", "accuracy": "accuracy"},
    refit="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

search.fit(X_train, y_train)

print("Best CV ROC AUC:", round(search.best_score_, 4))
print("Best parameters:")
search.best_params_

Best CV ROC AUC: 0.8409
Best parameters:


{'model__colsample_bytree': 0.8,
 'model__learning_rate': 0.1,
 'model__max_depth': 2,
 'model__n_estimators': 50,
 'model__reg_lambda': 5,
 'model__scale_pos_weight': 1,
 'model__subsample': 0.8}

In [7]:
cv_results = pd.DataFrame(search.cv_results_)

ranked_results = (
    cv_results[
        [
            "rank_test_roc_auc",
            "mean_test_roc_auc",
            "std_test_roc_auc",
            "mean_test_f1",
            "mean_test_accuracy",
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__learning_rate",
            "param_model__subsample",
            "param_model__colsample_bytree",
            "param_model__reg_lambda",
            "param_model__scale_pos_weight",
        ]
    ]
    .sort_values("rank_test_roc_auc")
    .head(10)
)

ranked_results

,rank_test_roc_auc,mean_test_roc_auc,std_test_roc_auc,mean_test_f1,mean_test_accuracy,param_model__n_estimators,param_model__max_depth,param_model__learning_rate,param_model__subsample,param_model__colsample_bytree,param_model__reg_lambda,param_model__scale_pos_weight
148,1,0.840950,0.016383,0.651166,0.778542,50,2,0.10,0.8,0.8,5,1.000000
150,2,0.840752,0.017097,0.693554,0.765440,50,2,0.10,0.8,0.8,5,1.869159
86,3,0.839846,0.017096,0.685409,0.760602,100,2,0.05,0.8,0.8,5,1.869159
18,4,0.839625,0.017490,0.694253,0.768706,200,2,0.01,0.8,0.8,1,1.869159
80,5,0.839546,0.021239,0.644652,0.776863,100,2,0.05,0.8,0.8,1,1.000000
84,6,0.838963,0.015342,0.640876,0.771985,100,2,0.05,0.8,0.8,5,1.000000
147,7,0.838670,0.020623,0.690699,0.763814,50,2,0.10,1.0,0.8,1,1.869159
366,8,0.838594,0.015702,0.682512,0.758936,50,2,0.10,0.8,1.0,5,1.869159
87,9,0.838314,0.016271,0.672951,0.755724,100,2,0.05,1.0,0.8,5,1.869159
302,10,0.838110,0.018855,0.679528,0.760589,100,2,0.05,0.8,1.0,5,1.869159


## Held-out test performance

The default 0.50 threshold is reported, and a second threshold is selected on the training set to maximize F1. The test set remains held out until this final evaluation.

In [8]:
best_model = search.best_estimator_

y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_test_proba = best_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_train_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]

metrics = pd.DataFrame(
    {
        "threshold": [0.50, best_threshold],
        "roc_auc": [roc_auc_score(y_test, y_test_proba)] * 2,
        "accuracy": [
            accuracy_score(y_test, y_test_proba >= 0.50),
            accuracy_score(y_test, y_test_proba >= best_threshold),
        ],
        "f1": [
            f1_score(y_test, y_test_proba >= 0.50),
            f1_score(y_test, y_test_proba >= best_threshold),
        ],
    },
    index=["default_0.50", "train_f1_optimized"],
)

metrics

,threshold,roc_auc,accuracy,f1
default_0.50,0.500000,0.816296,0.733766,0.577320
train_f1_optimized,0.333893,0.816296,0.727273,0.666667


In [9]:
test_predictions = (y_test_proba >= best_threshold).astype(int)

print(f"Selected threshold: {best_threshold:.3f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=3))

Selected threshold: 0.334

Confusion matrix:
[[70 30]
 [12 42]]

Classification report:
              precision    recall  f1-score   support

           0      0.854     0.700     0.769       100
           1      0.583     0.778     0.667        54

    accuracy                          0.727       154
   macro avg      0.718     0.739     0.718       154
weighted avg      0.759     0.727     0.733       154



## Feature importance

XGBoost importances show which features contributed most to the fitted boosted trees.

In [10]:
final_xgb = best_model.named_steps["model"]

importance_table = pd.DataFrame(
    {
        "feature": X.columns,
        "importance": final_xgb.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_table

,feature,importance
1,Glucose,0.311757
5,BMI,0.178198
4,Insulin,0.143845
7,Age,0.108095
0,Pregnancies,0.091929
6,DiabetesPedigreeFunction,0.083757
3,SkinThickness,0.053220
2,BloodPressure,0.029201


In [11]:
print("Final leakage-safe XGBoost pipeline:")
print(best_model)

Final leakage-safe XGBoost pipeline:
Pipeline(steps=[('zero_imputer',
                 ZeroToMedianImputer(columns=['Glucose', 'BloodPressure',
                                              'SkinThickness', 'Insulin',
                                              'BMI'])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_t...ights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                     